# Data warehousing architecture

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/concepts/data-warehousing.ipynb)

Warehouse layers, dimensional modeling, slowly changing dimensions, loading patterns, and testing. The SQL examples run
in-process on DuckDB, so every pattern below executes as written.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.0"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import duckdb

con = duckdb.connect()  # in-memory warehouse shared by every example below


def show(sql: str) -> list[tuple]:
    """Run a query, print it as a small table, and return the rows."""
    relation = con.sql(sql)
    relation.show()
    return relation.fetchall()

## Architecture and layers

### What are the layers of a data warehouse?

Each layer changes the data in exactly one way, so a bug can be fixed by rebuilding one layer instead of re-reading
the source.

```text
source -> landing/raw -> staging -> ODS -> warehouse (core) -> marts -> BI
          append-only    typed,     current   conformed dims    aggregated,
          replayable     deduped    entities  atomic facts      per team
```

+ **Landing / raw (bronze)**: an immutable copy of what the source sent, so a bad parse can be replayed.
+ **Staging (silver)**: typed and deduplicated; the only place raw payloads are parsed.
+ **ODS**: current-state operational entities, integrated across systems, little history.
+ **Warehouse / integration**: business rules, conformed dimensions, atomic facts, history.
+ **Presentation / marts (gold)**: star schemas, aggregates, and the semantic layer for one team or tool.

Worked example: an order arrives twice because of a retry. Landing keeps both rows, staging deduplicates on
`order_id` and the update timestamp, the ODS shows the latest status, the warehouse keeps the order fact, and the mart
shows today's open orders.

Common mistake: business joins in staging. A rule change then forces a re-read of raw data instead of a rebuild of
one model layer.

The classic **three-tier** description maps onto the same layers: bottom tier = sources, ingestion, and storage;
middle tier = integration, warehouse, and semantic model; top tier = BI, ad hoc SQL, and APIs.

### How do OLTP and OLAP systems differ?

| | OLTP | OLAP |
|---|---|---|
| Workload | many small reads/writes (place an order) | large scans and aggregations (revenue by month) |
| Model | normalized (3NF), ER modeling | dimensional (star/snowflake) |
| Storage | row-oriented, B-tree indexes | columnar, partition pruning, zone maps |
| Priority | low-latency writes, consistency | scan throughput, concurrency of reads |

Data moves from OLTP to OLAP with batch extracts, CDC, or event streams.

### How fresh can a warehouse be?

| Mode | Freshness | Typical stack | Cost model |
|---|---|---|---|
| Batch | hours to a day | nightly ELT | per run |
| Micro-batch | 1-15 minutes | Spark Structured Streaming, Auto Loader | per run, many runs |
| Near real time | seconds | Kafka + Flink + a fast store | per running hour |
| Real time | sub-second | streaming engine + serving layer | per running hour |

A true real-time warehouse fights the reason a warehouse exists (isolating analytics from operations) and makes
surrogate-key resolution and aggregate maintenance hard. Near real time is the usual compromise: frequent selective
refreshes of the tables that change, with full rebuilds further apart. Every streaming path needs an event-time
watermark, a dedup key, and a batch reconciliation, or it drifts away from the warehouse.

### Kimball, Inmon, or Data Vault?

| Approach | First artifact | Strength | Cost |
|---|---|---|---|
| Kimball (bottom-up) | dimensional mart per business process, integrated by conformed dimensions and a bus matrix | fast time to value, simple queries | integration discipline needed later |
| Inmon (top-down) | normalized (3NF) enterprise warehouse: subject-oriented, integrated, non-volatile, time-variant; marts derived from it | one version of the truth | long initial build |
| Data Vault | hubs (business keys), links (relationships), satellites (descriptive history), insert-only | auditability, onboarding many changing sources | more tables; needs marts on top for BI |

They compose rather than compete: many mature platforms run a Data Vault integration layer under Kimball marts. Use
Data Vault when many sources arrive incrementally and auditors need immutable history with provenance; skip it for a
small, stable set of sources.

```sql
-- Data Vault: satellites insert a row only when the descriptive hash changes.
INSERT INTO sat_customer
SELECT s.customer_hk, s.load_ts, s.hash_diff, s.name, s.city, s.segment
FROM staging.customer s
LEFT JOIN sat_customer c ON c.customer_hk = s.customer_hk AND c.hash_diff = s.hash_diff
WHERE c.customer_hk IS NULL;
```

### What are the medallion architecture and the lakehouse?

**Medallion** is the lakehouse name for the same layering: bronze (raw, append-only, replayable) -> silver (typed,
deduplicated, conformed keys) -> gold (star schemas, marts, aggregates). Rules: bronze is append-only so anything
downstream can be rebuilt; only silver parses and deduplicates; each layer has an owner, a freshness SLO, and a schema
contract; never land straight into gold.

A **lakehouse** is object storage + an open table format (Delta, Iceberg, Hudi) + a catalog: warehouse semantics on
lake storage.

| Aspect | Data warehouse | Lakehouse |
|---|---|---|
| Storage | proprietary, managed | open files (Parquet) plus table format |
| Compute | coupled to one engine | pluggable (Spark, Trino, Flink, SQL) |
| Transactions | native | table-format ACID, snapshots, time travel |
| Data types | structured, some semi-structured | structured, semi- and unstructured |
| Governance | mature RBAC and lineage | needs a catalog |

Choose a lakehouse for open formats and many engines; choose a managed warehouse for high-concurrency BI with less
operational overhead. The format adds reliability to storage, not modeling: grain, keys, and tests still apply.

### What are data marts, ODS, and virtual warehouses?

+ **Data mart**: a subject-area subset of the warehouse (finance, loans) for one team or tool; a different scope, not
  a different technology. **Dependent** marts are fed from the warehouse and inherit its conformed dimensions;
  **independent** marts read sources directly and drift into a second definition of revenue.
+ **ODS (operational data store)**: integrated, current-state, near-real-time operational data with short history.
  A mart can be built from an ODS when it only needs recent operational data, but historical analysis needs the
  warehouse.
+ **Virtual data warehouse**: users query operational databases directly (federation) instead of a physical copy.
  Cheap to start; slow and risky for the source systems at scale.

## Dimensional modeling

### What is dimensional modeling, and what is the grain?

A dimensional model is a fact table at a declared grain surrounded by dimensions that carry descriptive context.
The four-step design process:

1. Choose the business process (retail sales).
2. Declare the grain: the sentence "one row per ___" (one row per order line).
3. Identify the dimensions that describe that grain (date, product, store, customer).
4. Identify the numeric facts that are true at that grain (quantity, net amount).

Store the finest (atomic) grain you can afford: aggregates can be derived from it, but a coarse fact can never be
drilled back to detail you did not store. Never mix grains in one table: loading order-header totals next to order
lines double-counts the header when you sum.

**Fact** = a measurement of a business process. **Dimension** = the context you group and filter by. **Attribute** =
a descriptive column of a dimension (product name and category are attributes of the product dimension).

### Star schema versus snowflake schema

A star keeps each dimension denormalized, one join from the fact. A snowflake normalizes hierarchies into
sub-dimension tables, adding a join per level. The cell builds both on the same data and proves they give the same
answer.

In [ ]:
con.sql("""
CREATE TABLE dim_date AS
SELECT CAST(strftime(d, '%Y%m%d') AS INTEGER) AS date_sk, CAST(d AS DATE) AS date,
       year(d) AS year, quarter(d) AS quarter, month(d) AS month
FROM generate_series(DATE '2026-01-01', DATE '2026-01-31', INTERVAL 1 DAY) AS t(d);

-- Star: category and subcategory live on the product dimension.
CREATE TABLE dim_product (product_sk INTEGER PRIMARY KEY, product_id TEXT UNIQUE,
                          product_name TEXT, subcategory TEXT, category TEXT, unit_cost DECIMAL(10, 2));
INSERT INTO dim_product VALUES
    (1, 'P-1', 'Laptop',  'Computers', 'Electronics', 700),
    (2, 'P-2', 'Monitor', 'Displays',  'Electronics', 120),
    (3, 'P-3', 'Desk',    'Office',    'Furniture',   150);

CREATE TABLE dim_store (store_sk INTEGER PRIMARY KEY, store_id TEXT UNIQUE,
                        country TEXT, region TEXT, city TEXT);
INSERT INTO dim_store VALUES
    (1, 'S-1', 'IN', 'South', 'Bangalore'),
    (2, 'S-2', 'IN', 'South', 'Hyderabad'),
    (3, 'S-3', 'IN', 'North', 'Delhi');

-- Grain: one row per order line. sale_id is a degenerate dimension.
CREATE TABLE fct_sales (sale_id INTEGER PRIMARY KEY, date_sk INTEGER, product_sk INTEGER,
                        store_sk INTEGER, quantity INTEGER, net_amount DECIMAL(12, 2));
INSERT INTO fct_sales VALUES
    (101, 20260105, 1, 1, 1, 1000), (102, 20260105, 2, 1, 2, 400),
    (103, 20260112, 3, 2, 1, 250),  (104, 20260120, 1, 3, 2, 2000),
    (105, 20260128, 2, 2, 1, 200);

-- Snowflake: the product hierarchy is normalized into two lookup tables.
CREATE TABLE dim_category (category_sk INTEGER PRIMARY KEY, category TEXT);
CREATE TABLE dim_subcategory (subcategory_sk INTEGER PRIMARY KEY, subcategory TEXT, category_sk INTEGER);
CREATE TABLE dim_product_sf (product_sk INTEGER PRIMARY KEY, product_name TEXT, subcategory_sk INTEGER);
INSERT INTO dim_category VALUES (1, 'Electronics'), (2, 'Furniture');
INSERT INTO dim_subcategory VALUES (1, 'Computers', 1), (2, 'Displays', 1), (3, 'Office', 2);
INSERT INTO dim_product_sf VALUES (1, 'Laptop', 1), (2, 'Monitor', 2), (3, 'Desk', 3);
""")

star = show("""
SELECT p.category, SUM(f.net_amount) AS revenue
FROM fct_sales f
JOIN dim_product p ON p.product_sk = f.product_sk          -- one join
GROUP BY p.category ORDER BY p.category
""")
snowflake = con.execute("""
SELECT c.category, SUM(f.net_amount) AS revenue
FROM fct_sales f
JOIN dim_product_sf p   ON p.product_sk = f.product_sk     -- three joins for the same answer
JOIN dim_subcategory s  ON s.subcategory_sk = p.subcategory_sk
JOIN dim_category c     ON c.category_sk = s.category_sk
GROUP BY c.category ORDER BY c.category
""").fetchall()

assert star == snowflake == [("Electronics", 3600), ("Furniture", 250)]

Default to a star for marts and BI: simpler for users and tools, and on columnar MPP engines the extra joins cost
more than the storage saved. Snowflake a dimension only when a sub-dimension is large, shared, maintained by a
different owner, or changes independently. Common mistake: snowflaking a small dimension for tidiness and paying for
the extra joins on every dashboard query.

Related terms: a **galaxy schema** (fact constellation) is several fact tables sharing dimensions; a **cube** is the
logical multidimensional view of a star (dimension members on the edges, measures in the cells).

### Measures: additive, semi-additive, non-additive

+ **Additive**: sums across every dimension (units sold).
+ **Semi-additive**: sums across some dimensions but not time (account balance, units on hand). Use the period-end
  value or an average over time.
+ **Non-additive**: never sums (ratios, percentages, distinct counts). Store the additive components and recompute.

In [ ]:
con.sql("""
-- Periodic snapshot: one row per product per store per day.
CREATE TABLE fct_inventory_daily (date_sk INTEGER, product_sk INTEGER, store_sk INTEGER, units_on_hand INTEGER);
INSERT INTO fct_inventory_daily VALUES
    (20260129, 1, 1, 10), (20260129, 2, 1, 4),
    (20260130, 1, 1,  8), (20260130, 2, 1, 4),
    (20260131, 1, 1,  7), (20260131, 2, 1, 3);
""")

wrong_total = con.sql("SELECT SUM(units_on_hand) FROM fct_inventory_daily").fetchall()[0][0]
month_end = show("""
SELECT SUM(units_on_hand) AS units_on_hand_month_end
FROM fct_inventory_daily
WHERE date_sk = (SELECT MAX(date_sk) FROM fct_inventory_daily)
""")[0][0]
assert (wrong_total, month_end) == (36, 10)  # 36 units never existed; 10 is the real month-end stock

# Non-additive: average the ratio vs recompute it from summed components.
margins = show("""
WITH lines AS (
    SELECT f.net_amount AS revenue, f.net_amount - f.quantity * p.unit_cost AS margin
    FROM fct_sales f JOIN dim_product p USING (product_sk)
)
SELECT ROUND(AVG(margin / revenue), 4) AS avg_of_ratios,
       ROUND(SUM(margin) / SUM(revenue), 4) AS ratio_of_sums
FROM lines
""")[0]
assert margins == (0.36, 0.3221)  # only ratio_of_sums is the true margin percent

### What types of fact tables are there?

| Type | Grain | Example | Notes |
|---|---|---|---|
| Transaction | one row per event | one row per sale | default; most flexible |
| Periodic snapshot | one row per entity per period | account balance per day | measures are usually semi-additive |
| Accumulating snapshot | one row per process instance, updated as it advances | order with ordered/shipped/delivered dates | answers latency questions (days to ship) |
| Factless | dimension keys only; `COUNT(*)` is the measure | promotion coverage, attendance | records events or eligibility |

The fact type decides what "late data" means: a late transaction is appended to its partition, a late snapshot
restates a period, and a late milestone updates an accumulating row.

### Surrogate keys

A surrogate key is a warehouse-generated, meaningless integer that identifies one dimension row. Natural (source)
keys are not enough: sources merge, reuse, or reformat them, and a Type 2 dimension needs many rows per natural key.
Facts store the surrogate, so each fact resolves to the exact dimension version that was true at load time. Integer
surrogates also make joins and indexes smaller. Assign them in one place: two pipelines generating keys independently
split one customer's history in two. The SCD section below shows surrogates in action.

### Conformed dimensions, conformed facts, and the bus matrix

A **conformed dimension** has identical keys, column names, and values in every fact table that uses it; a
**conformed fact** has the same name, grain, and business rule wherever it appears. Together they make **drilling
across** possible: combining measures from different fact tables on shared dimensions. The **bus matrix** (Kimball's
bus architecture) plans them: business processes as rows, conformed dimensions as columns.

```text
process     date  customer  product  store
sales        x       x         x       x
returns      x       x         x       x
inventory    x                 x       x
```

Common mistake: `dim_date_sales` and `dim_date_finance` with different fiscal calendars and no mapping, so the marts
cannot be compared.

In [ ]:
con.execute("""
CREATE TABLE fct_returns (return_id INTEGER, date_sk INTEGER, product_sk INTEGER, store_sk INTEGER,
                          refund_amount DECIMAL(12, 2));
INSERT INTO fct_returns VALUES (1, 20260115, 2, 1, 200), (2, 20260129, 1, 3, 1000);
""")

# Drill across: aggregate each fact separately to the shared grain, then join on the conformed dimension.
drill = show("""
WITH sales AS (
    SELECT p.category, SUM(net_amount) AS revenue FROM fct_sales JOIN dim_product p USING (product_sk) GROUP BY 1
), returns AS (
    SELECT p.category, SUM(refund_amount) AS refunds FROM fct_returns JOIN dim_product p USING (product_sk) GROUP BY 1
)
SELECT category, revenue, COALESCE(refunds, 0) AS refunds, revenue - COALESCE(refunds, 0) AS net
FROM sales LEFT JOIN returns USING (category)
ORDER BY category
""")
assert drill == [("Electronics", 3600, 1200, 2400), ("Furniture", 250, 0, 250)]

Aggregate each fact to the shared grain before joining. Joining two fact tables row to row fans out and multiplies
both measures.

### Junk, degenerate, and factless

+ **Junk dimension**: low-cardinality flags and indicators (`is_gift`, `payment_type`, `channel`) collected into one
  small dimension of their combinations, instead of cluttering the fact or creating many tiny dimensions.
+ **Degenerate dimension**: an identifier kept on the fact with no dimension table (order number, invoice id). It is
  part of the grain and is used for grouping order lines.
+ **Factless fact table**: only dimension keys. It answers coverage questions measures cannot, such as which promoted
  products did not sell.

In [ ]:
con.execute("""
CREATE TABLE dim_promotion (promotion_sk INTEGER PRIMARY KEY, promotion_name TEXT);
INSERT INTO dim_promotion VALUES (1, 'New Year'), (2, 'Clearance');

-- Factless fact: which products were on promotion, whether or not they sold.
CREATE TABLE fct_promotion_coverage (date_sk INTEGER, product_sk INTEGER, store_sk INTEGER, promotion_sk INTEGER);
INSERT INTO fct_promotion_coverage VALUES
    (20260105, 1, 1, 1), (20260105, 2, 1, 1), (20260105, 3, 1, 1);
""")

unsold = show("""
SELECT p.product_name AS promoted_but_unsold
FROM fct_promotion_coverage c
JOIN dim_product p USING (product_sk)
WHERE NOT EXISTS (
    SELECT 1 FROM fct_sales s
    WHERE s.product_sk = c.product_sk AND s.store_sk = c.store_sk AND s.date_sk = c.date_sk
)
""")
assert unsold == [("Desk",)]

### Bridge tables for many-to-many relationships

When one fact row relates to many dimension rows (an order with several promotions, a hospital visit with several
diagnoses), put a bridge table between them with a **weight** per pairing. Weights for one fact row sum to 1 (use
`1 / count` when no business rule exists), so allocated measures add back to the true total.

In [ ]:
con.execute("""
CREATE TABLE bridge_order_promotion (sale_id INTEGER, promotion_sk INTEGER, weight DECIMAL(6, 4));
INSERT INTO bridge_order_promotion VALUES
    (101, 1, 0.5), (101, 2, 0.5),   -- two promotions share order 101
    (104, 1, 1.0);
""")

naive = con.execute("""
SELECT SUM(f.net_amount) FROM fct_sales f JOIN bridge_order_promotion b USING (sale_id)
""").fetchall()[0][0]
allocated = show("""
SELECT p.promotion_name, SUM(f.net_amount * b.weight) AS allocated_revenue
FROM fct_sales f
JOIN bridge_order_promotion b USING (sale_id)
JOIN dim_promotion p USING (promotion_sk)
GROUP BY p.promotion_name ORDER BY p.promotion_name
""")
true_total = 1000 + 2000  # orders 101 and 104
assert naive == 4000  # order 101 counted twice without weights
assert sum(revenue for _, revenue in allocated) == true_total

### Hierarchies, ROLLUP, and GROUPING SETS

A hierarchy is an ordered set of levels inside a dimension (country -> region -> city, year -> quarter -> month).
Storing every level on the dimension row (star style) makes drill-down a `GROUP BY`. `ROLLUP` produces subtotals along
the hierarchy, `GROUPING SETS` picks arbitrary combinations, and `GROUPING()` tells a subtotal row from a real NULL.
Precomputing these combinations is what OLAP aggregations are; choosing which ones to materialize is the
**view selection problem**.

In [ ]:
rollup = show("""
SELECT s.region, s.city, SUM(f.net_amount) AS revenue, GROUPING(s.region, s.city) AS level
FROM fct_sales f JOIN dim_store s USING (store_sk)
GROUP BY ROLLUP (s.region, s.city)
ORDER BY level, s.region, s.city
""")
grand_total = [revenue for region, city, revenue, level in rollup if level == 3]
assert grand_total == [3850]
assert {(region, revenue) for region, city, revenue, level in rollup if level == 1} == {("North", 2000), ("South", 1850)}

Distinct counts do not re-aggregate from a coarser table: keep them at the detail grain or use a sketch such as
HyperLogLog.

### Generating the date dimension

A date dimension is generated, not loaded from a source: one row per day for a fixed range, with precomputed
attributes (fiscal periods, weekends, holidays) so queries never apply functions to the fact's date column. Integer
`YYYYMMDD` keys are readable and sort correctly.

In [ ]:
con.execute("""
CREATE TABLE dim_date_full AS
SELECT CAST(strftime(d, '%Y%m%d') AS INTEGER)          AS date_sk,
       CAST(d AS DATE)                                 AS date,
       year(d)                                         AS year,
       quarter(d)                                      AS quarter,
       strftime(d, '%b')                               AS month_name,
       isodow(d) IN (6, 7)                             AS is_weekend,
       CASE WHEN month(d) >= 4 THEN year(d) ELSE year(d) - 1 END AS fiscal_year  -- April fiscal start
FROM generate_series(DATE '2020-01-01', DATE '2030-12-31', INTERVAL 1 DAY) AS t(d)
""")
rows, weekends = con.sql("SELECT COUNT(*), COUNT(*) FILTER (WHERE is_weekend) FROM dim_date_full").fetchall()[0]
assert rows == 4018  # 11 years including 3 leap days
assert con.sql("SELECT fiscal_year FROM dim_date_full WHERE date = DATE '2026-02-01'").fetchall()[0][0] == 2025
print(f"{rows} days, {weekends} weekend days")

### Conceptual, logical, and physical models

| Model | Shows | Keys | Audience |
|---|---|---|---|
| Conceptual | main entities and that they relate | none | business |
| Logical | entities, all attributes, relationships, many-to-many resolved, normalized | primary and foreign keys | architects |
| Physical | tables, columns, data types, indexes, partitions, denormalization; specific to one engine | enforced or declared constraints | engineers |

ER modeling normalizes for OLTP writes; dimensional modeling deliberately denormalizes dimensions for reads. A fact
table is still close to normalized: it holds foreign keys plus measures that depend on the full key.

## Slowly changing dimensions

### Which SCD type should you use?

The SCD type answers one question: what do we keep when an attribute changes?

| Type | Storage | History | Typical use |
|---|---|---|---|
| 0 | never changes | original value only | date of birth, original signup channel |
| 1 | overwrite in place | none | correcting a typo |
| 2 | new row per version, validity dates, current flag | full | segment, address, price band |
| 3 | extra `previous_*` column | one prior value | a single before/after comparison |
| 4 | current table + separate history table | full, out of band | fast-changing attributes |
| 6 (1+2+3) | Type 2 rows plus a `*_current` column on every row | full, easy current filter | report by then-true and current value |

Choose per attribute, not per table: a **hybrid** dimension overwrites some columns (Type 1) and versions others
(Type 2). Type 2 on everything grows the dimension with no business reason; Type 2 on a typo invents fake history.

In [ ]:
con.execute("""
CREATE SEQUENCE customer_sk_seq START 1;
CREATE TABLE dim_customer (
    customer_sk     INTEGER PRIMARY KEY DEFAULT nextval('customer_sk_seq'),
    customer_id     TEXT NOT NULL,          -- natural key: repeats once per version
    name            TEXT NOT NULL,          -- Type 1: corrected in place
    city            TEXT NOT NULL,          -- Type 2: history matters
    start_date      DATE NOT NULL,
    end_date        DATE NOT NULL DEFAULT DATE '9999-12-31',
    is_current      BOOLEAN NOT NULL DEFAULT TRUE
);
INSERT INTO dim_customer (customer_id, name, city, start_date) VALUES
    ('C-100', 'Acme Ltd',  'Bangalore', DATE '2025-01-01'),
    ('C-200', 'Bolt Corp', 'Delhi',     DATE '2025-01-01');
CREATE TABLE customer_delta (customer_id TEXT, name TEXT, city TEXT, effective_date DATE);
""")


def load_customer_delta(rows: list[tuple]) -> None:
    """Apply a delta: Type 1 for name, Type 2 for city. Safe to re-run with the same rows."""
    con.execute("DELETE FROM customer_delta")
    con.executemany("INSERT INTO customer_delta VALUES (?, ?, ?, ?)", rows)
    con.execute("""
    -- Type 1: overwrite the name on every version of the customer.
    UPDATE dim_customer d SET name = s.name
    FROM customer_delta s
    WHERE d.customer_id = s.customer_id AND d.name IS DISTINCT FROM s.name;

    -- Type 2 step 1: close the current version when a tracked attribute changed.
    UPDATE dim_customer d
    SET end_date = s.effective_date - INTERVAL 1 DAY, is_current = FALSE
    FROM customer_delta s
    WHERE d.customer_id = s.customer_id AND d.is_current AND d.city IS DISTINCT FROM s.city;

    -- Type 2 step 2: insert a new current version for changed and brand-new customers.
    INSERT INTO dim_customer (customer_id, name, city, start_date)
    SELECT s.customer_id, s.name, s.city, s.effective_date
    FROM customer_delta s
    WHERE NOT EXISTS (SELECT 1 FROM dim_customer c WHERE c.customer_id = s.customer_id AND c.is_current);
    """)


delta = [
    ("C-100", "Acme Ltd",       "Hyderabad", "2025-07-01"),  # moved: Type 2
    ("C-200", "Bolt Corporation", "Delhi",   "2025-07-01"),  # renamed: Type 1
    ("C-300", "Cyan Inc",       "Mumbai",    "2025-07-01"),  # new customer
]
load_customer_delta(delta)
load_customer_delta(delta)  # re-running the same delta must not add versions

versions = show("SELECT * FROM dim_customer ORDER BY customer_id, start_date")
assert len(versions) == 4
assert con.sql("SELECT COUNT(*) FROM dim_customer WHERE customer_id = 'C-100'").fetchall()[0][0] == 2
assert con.sql("SELECT name FROM dim_customer WHERE customer_id = 'C-200'").fetchall() == [("Bolt Corporation",)]

Notes on the pattern:

+ `IS DISTINCT FROM` is NULL-safe; `<>` silently misses changes to or from NULL.
+ Guarding with `is_current` makes the load idempotent: the second run above changed nothing.
+ The same logic can be written as one `MERGE`, but a single MERGE cannot close the old row and insert the new one for
  the same key in most engines. Either union the changed rows into the source twice (once to update, once with a NULL
  merge key to insert) or keep the two-step form.
+ Production loads also need `UNIQUE (customer_id, start_date)`, `CHECK (end_date > start_date)`, and one surrogate
  key generator.

### Point-in-time fact joins

Facts must store the surrogate key of the version that was valid **when the event happened**, so revenue stays
attributed to the city the customer lived in at the time.

In [ ]:
con.execute("""
CREATE TABLE staging_orders (order_id INTEGER, customer_id TEXT, order_date DATE, net_amount DECIMAL(12, 2));
INSERT INTO staging_orders VALUES
    (1, 'C-100', DATE '2025-03-10', 500),   -- while in Bangalore
    (2, 'C-100', DATE '2025-09-15', 300),   -- after moving to Hyderabad
    (3, 'C-300', DATE '2025-08-01', 150);

CREATE TABLE fct_orders AS
SELECT o.order_id, o.order_date, d.customer_sk, o.net_amount
FROM staging_orders o
JOIN dim_customer d
  ON d.customer_id = o.customer_id
 AND o.order_date BETWEEN d.start_date AND d.end_date;
""")

by_city = show("""
SELECT d.city, SUM(f.net_amount) AS revenue
FROM fct_orders f JOIN dim_customer d USING (customer_sk)
GROUP BY d.city ORDER BY d.city
""")
assert by_city == [("Bangalore", 500), ("Hyderabad", 300), ("Mumbai", 150)]

### Type 3 and Type 6

Type 3 adds a `previous_city` column: cheap, but only one prior value survives. Type 6 keeps full Type 2 history and
adds a `city_current` column overwritten on every version, so one table answers both "revenue by the city at the
time" and "revenue by where customers live now".

In [ ]:
con.execute("""
ALTER TABLE dim_customer ADD COLUMN city_current TEXT;

-- Type 6: copy the current value onto every version of the natural key.
UPDATE dim_customer d SET city_current = c.city
FROM dim_customer c
WHERE c.customer_id = d.customer_id AND c.is_current;
""")

now_vs_then = show("""
SELECT d.city AS city_at_order, d.city_current, SUM(f.net_amount) AS revenue
FROM fct_orders f JOIN dim_customer d USING (customer_sk)
WHERE d.customer_id = 'C-100'
GROUP BY ALL ORDER BY city_at_order
""")
assert now_vs_then == [("Bangalore", "Hyderabad", 500), ("Hyderabad", "Hyderabad", 300)]

## Loading the warehouse

### ETL versus ELT

ETL transforms on a separate server before loading; ELT loads raw data first and transforms inside the warehouse with
SQL models. ELT wins on scalable cloud warehouses because raw data stays replayable: a wrong join means rebuilding one
model, not re-extracting from the source. ETL still fits constrained targets or data that must be masked before it
lands. ELT moves the work into SQL, so layering, tests, orchestration, and idempotency matter just as much.

### Staging: type and deduplicate

Staging keeps the latest version of each business key with `ROW_NUMBER()` (or DuckDB/Snowflake/BigQuery `QUALIFY`).

In [ ]:
con.execute("""
CREATE TABLE landing_orders (order_id INTEGER, status TEXT, amount TEXT, updated_at TIMESTAMP);
INSERT INTO landing_orders VALUES
    (1, ' paid ',   '10.50', TIMESTAMP '2026-01-05 10:00'),
    (1, 'shipped',  '10.50', TIMESTAMP '2026-01-06 09:00'),   -- later version of order 1
    (1, 'shipped',  '10.50', TIMESTAMP '2026-01-06 09:00'),   -- exact duplicate from a retry
    (2, 'paid',     '99.00', TIMESTAMP '2026-01-06 11:00');

CREATE TABLE stg_orders AS
SELECT order_id, UPPER(TRIM(status)) AS status, CAST(amount AS DECIMAL(12, 2)) AS amount, updated_at
FROM landing_orders
QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC) = 1;
""")
staged = show("SELECT * FROM stg_orders ORDER BY order_id")
assert [(row[0], row[1]) for row in staged] == [(1, "SHIPPED"), (2, "PAID")]

### Full load versus incremental load

A **full load** truncates and rebuilds the target (simple, right for small dimensions; build a new table and swap it
atomically). An **incremental load** applies only changed rows since a stored **watermark** and must be idempotent: an
upsert on the natural key, or delete-the-partition-then-insert. A plain append is not idempotent: a retry duplicates
the partition.

(Database-specific "conventional" versus "direct-path" loads are the same trade-off at the engine level: validate
constraints while inserting, or disable them for speed and validate afterwards.)

In [ ]:
con.execute("""
CREATE TABLE fct_order_status (order_id INTEGER PRIMARY KEY, status TEXT, amount DECIMAL(12, 2),
                               updated_at TIMESTAMP);
CREATE TABLE etl_watermark (table_name TEXT PRIMARY KEY, high_water TIMESTAMP);
INSERT INTO etl_watermark VALUES ('fct_order_status', TIMESTAMP '1900-01-01');
""")

UPSERT = """
INSERT INTO fct_order_status
SELECT order_id, status, amount, updated_at FROM stg_orders
WHERE updated_at > (SELECT high_water FROM etl_watermark WHERE table_name = 'fct_order_status')
ON CONFLICT (order_id) DO UPDATE
    SET status = excluded.status, amount = excluded.amount, updated_at = excluded.updated_at
    WHERE excluded.updated_at > fct_order_status.updated_at;

UPDATE etl_watermark
SET high_water = (SELECT MAX(updated_at) FROM fct_order_status)
WHERE table_name = 'fct_order_status';
"""
con.execute(UPSERT)
con.execute(UPSERT)  # a retry reads nothing new and changes nothing

con.execute("INSERT INTO stg_orders VALUES (2, 'REFUNDED', 99.00, TIMESTAMP '2026-01-07 08:00')")
con.execute(UPSERT)  # the next run picks up only the new change
final = show("SELECT order_id, status, updated_at FROM fct_order_status ORDER BY order_id")
assert [(order_id, status) for order_id, status, _ in final] == [(1, "SHIPPED"), (2, "REFUNDED")]

### Late-arriving facts and dimensions

A **late fact** arrives after its period was loaded: write it into its event-date partition, then recompute only the
affected aggregate partitions, within an agreed window (for example 7 days); older arrivals are approved corrections.
A **late dimension** (early-arriving fact) references a customer the dimension does not have yet: never drop the fact.
Point it at an inferred or unknown member, keep the natural key, and repair the key when the real row arrives.

In [ ]:
con.execute("""
INSERT INTO dim_customer (customer_sk, customer_id, name, city, start_date, is_current)
VALUES (-1, 'UNKNOWN', 'Unknown customer', 'Unknown', DATE '1900-01-01', TRUE);

CREATE TABLE staging_sales (sale_id INTEGER, customer_id TEXT, sale_date DATE, net_amount DECIMAL(12, 2));
INSERT INTO staging_sales VALUES (900, 'C-400', DATE '2026-01-10', 75);   -- C-400 not in the dimension yet

CREATE TABLE fct_sales_cust AS
SELECT s.sale_id, s.customer_id, s.sale_date, COALESCE(d.customer_sk, -1) AS customer_sk, s.net_amount
FROM staging_sales s
LEFT JOIN dim_customer d ON d.customer_id = s.customer_id AND d.is_current;
""")
assert con.sql("SELECT customer_sk FROM fct_sales_cust").fetchall()[0][0] == -1  # kept, not dropped

# The dimension row arrives later: load it, then repair the fact keys.
load_customer_delta([("C-400", "Delta LLC", "Pune", "2026-01-01")])
con.execute("""
UPDATE fct_sales_cust f SET customer_sk = d.customer_sk
FROM dim_customer d
WHERE f.customer_sk = -1 AND d.customer_id = f.customer_id
  AND f.sale_date BETWEEN d.start_date AND d.end_date
""")
repaired = show("""
SELECT f.sale_id, d.customer_id, d.city FROM fct_sales_cust f JOIN dim_customer d USING (customer_sk)
""")
assert repaired == [(900, "C-400", "Pune")]

### Metadata, catalog, and lineage

Metadata is data about data. **Technical** metadata: schemas, types, partitions, freshness. **Business** metadata:
definitions, owners, grain. **Operational** metadata: load times, row counts, lineage. A data dictionary is the
subset that describes columns and valid values. The catalog records what exists and who owns it; column-level lineage
records where each column came from and what depends on it, which turns "what breaks if I change this column?" into
a query instead of an outage. A catalog must be generated from the running system, or it is stale within a sprint.

## Physical design and performance

### Partitioning, clustering, and indexing

+ **Partition** on the column nearly every query filters, usually the event date. Keep partitions large (hundreds of
  MB to a few GB); never partition on a high-cardinality key such as `customer_id` (millions of tiny files).
+ **Cluster / sort** inside partitions on the next most selective filters and join keys, so min/max statistics (zone
  maps) skip files.
+ **Indexes**: columnar warehouses rely on pruning, zone maps, and bloom filters rather than B-trees. Row-store
  warehouses use bitmap indexes on low-cardinality fact foreign keys and flags (Oracle; star-join bitmap AND/OR),
  and B-trees on dimension surrogate and natural keys.
+ **Cardinality** drives these choices: high (email), normal (last name), low (0/1 flags). Bitmaps suit low, B-trees
  suit high.

```sql
-- Snowflake: clustering key drives micro-partition pruning.
ALTER TABLE fct_sales CLUSTER BY (sale_date, store_sk);
-- Iceberg: change the partition spec without rewriting history.
ALTER TABLE fct_sales SET PARTITION SPEC (month(sale_date));
```

### Views, materialized views, and aggregate tables

A **view** stores only the query and recomputes it on every read. A **materialized view** stores the result and must
be refreshed (incrementally where the engine supports it). An **aggregate table** is a materialized summary at a
coarser grain built by the pipeline; it must be refreshed and tested like any other output.

### MOLAP, ROLAP, HOLAP

| Flavor | Storage | Trade-off |
|---|---|---|
| MOLAP | precomputed multidimensional cube | fastest queries, slow to build, limited detail |
| ROLAP | relational star schema, SQL at query time | scales to detail, slower without aggregates |
| HOLAP | detail in relational tables, aggregates in a cube | balance of both |

Modern columnar warehouses are effectively ROLAP with materialized views and caches.

## Testing a warehouse model

Test at four levels, and fail the run rather than logging a warning:

| Level | Example test | Fails when |
|---|---|---|
| Schema | column exists and type matches | the source changes shape |
| Grain | declared key is unique | a loader duplicates a row |
| Integrity | every foreign key resolves; one current row per natural key; no overlapping validity ranges | a late dimension row is missing |
| Business | revenue matches the source within a tolerance | a join silently drops rows |

The cell runs these checks against the model built above.

In [ ]:
CHECKS = {
    "fct_sales grain is unique": """
        SELECT sale_id FROM fct_sales GROUP BY sale_id HAVING COUNT(*) > 1""",
    "no orphan product keys": """
        SELECT f.sale_id FROM fct_sales f LEFT JOIN dim_product p USING (product_sk) WHERE p.product_sk IS NULL""",
    "one current row per customer": """
        SELECT customer_id FROM dim_customer WHERE is_current GROUP BY customer_id HAVING COUNT(*) <> 1""",
    "valid SCD ranges": """
        SELECT customer_sk FROM dim_customer WHERE end_date <= start_date""",
    "no overlapping versions": """
        SELECT a.customer_sk FROM dim_customer a JOIN dim_customer b
          ON a.customer_id = b.customer_id AND a.customer_sk < b.customer_sk
         AND a.start_date <= b.end_date AND b.start_date <= a.end_date""",
    "orders reconcile with staging": """
        SELECT 1 WHERE (SELECT SUM(net_amount) FROM fct_orders) <> (SELECT SUM(net_amount) FROM staging_orders)""",
}

for name, sql in CHECKS.items():
    failures = con.sql(sql).fetchall()
    print(f"{'PASS' if not failures else 'FAIL'}  {name}")
    assert not failures, f"{name}: {failures}"

## Designing a warehouse that scales

Four decisions, made in order:

1. **Architecture.** A layered warehouse when questions are known and governance is a hard requirement; a lakehouse
   when files are already the source of truth or one copy must serve many engines; a data mesh (domain-owned data
   products on a central platform) only when many teams own many sources and contracts plus a platform team exist.
   Central modeling gives one metric definition but is a bottleneck; decentralization gives speed but multiplies
   definitions. Pick one system of record per dataset.
2. **Ingestion.** Batch cost scales with runs, streaming cost with wall-clock hours (idle capacity at night).
   Micro-batch is the usual analytics default; hybrid adds a fast path plus a nightly reconciliation.
3. **Transformation.** ELT with orchestration (dependency order, retries, backfills), tests that fail the run, and
   idempotent loads. Schedule by data interval, not completion time, so backfills do not collide with live runs; reload
   a bounded late-data window instead of all history.
4. **Operations.** Publish a freshness SLA per table and measure event time versus availability, not job duration
   (a late start with a fast job is still stale). Watch the largest cost consumers and idle compute, capture
   column-level lineage for impact analysis, and prove RPO/RTO with restore drills.

```yaml
pipeline: daily_sales
schedule: "0 2 * * *"
tasks:
  - {id: stage_orders, depends_on: []}
  - {id: dim_customer_scd2, depends_on: [stage_orders]}
  - {id: fct_sales, depends_on: [dim_customer_scd2]}
  - {id: test_reconcile, depends_on: [fct_sales]}
```

## Things to remember

| Topic | Fact | Why it matters |
|---|---|---|
| Grain | "one row per ___" is decided before any column | every double-count bug is a grain bug |
| SCD types | 0 fixed, 1 overwrite, 2 new row with dates, 3 prior-value column, 6 = 2 + current column | Type 2 is the default answer and the expensive one |
| Star vs snowflake | star denormalizes dimensions | on MPP engines fewer joins win |
| Surrogate keys | warehouse-generated, decoupled from source keys | Type 2 is impossible without them |
| Conformed dimensions | the same dimension across marts | keeps metrics comparable; enables drill-across |
| Kimball vs Inmon | marts first via a bus vs normalized EDW first | Kimball ships faster; Inmon integrates first |
| Medallion | bronze -> silver -> gold | staging + warehouse + mart, restated for lakehouses |
| Fact types | transaction, periodic snapshot, accumulating snapshot, factless | the type decides what late data means |

Mistakes that keep coming back:

+ Declaring the grain after the columns exist.
+ Type 2 on every attribute by default (row explosion), or Type 2 on corrections (fake history).
+ Source primary keys as warehouse keys: a source re-key breaks history.
+ Joining two fact tables row to row instead of aggregating each to a shared grain first.
+ A "conformed" dimension that differs slightly per mart, so metrics diverge silently.
+ No unknown member for late dimensions: facts land with dangling keys and vanish from reports.
+ Appending in a retried load instead of upserting or replacing the partition.

## Real-world use case: clickstream sessionization

A new session starts after 30 minutes of inactivity. `LAG` finds the gap, a running `SUM` numbers the sessions, and the
result becomes a session fact that joins to the revenue fact on user and time.

In [ ]:
con.execute("""
CREATE TABLE click_events (user_id TEXT, event_ts TIMESTAMP);
INSERT INTO click_events VALUES
    ('u1', TIMESTAMP '2026-01-05 09:00'), ('u1', TIMESTAMP '2026-01-05 09:10'),
    ('u1', TIMESTAMP '2026-01-05 09:55'), ('u1', TIMESTAMP '2026-01-05 10:05'),
    ('u2', TIMESTAMP '2026-01-05 09:00');
""")
sessions = show("""
WITH flagged AS (
    SELECT *,
           CASE WHEN event_ts - LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts) <= INTERVAL 30 MINUTE
                THEN 0 ELSE 1 END AS is_new_session
    FROM click_events
), numbered AS (
    SELECT *, SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_ts) AS session_number
    FROM flagged
)
SELECT user_id, session_number, MIN(event_ts) AS session_start, MAX(event_ts) AS session_end, COUNT(*) AS events
FROM numbered
GROUP BY user_id, session_number
ORDER BY user_id, session_number
""")
assert [(user, number, events) for user, number, _, _, events in sessions] == [("u1", 1, 2), ("u1", 2, 2), ("u2", 1, 1)]

## Data warehouse exercises

Back-of-the-envelope sizing, cost, and correctness arithmetic in plain Python. The numbers check the formula, not a
real platform: compression, skew, caching, and concurrency add variance.

### Try it 1. Size storage from daily volume and retention

**Task.** A fact table receives 50,000,000 rows per day at 400 bytes per row before compression. The warehouse compresses at 4 to 1. Add 15 percent for indexes and metadata, keep two copies for redundancy, and retain 730 days. Compute the stored bytes per day and the total footprint in terabytes.

**Hint.** Convert to gigabytes first: `daily_rows * bytes_per_row / 1e9`. Then divide by compression, multiply by overhead and copies, and multiply by the retention days.

**Expected result.** `20.0 GB` raw per day, `11.5 GB` stored per day, and about `8395.0 GB` or `8.395` TB for the full retention.

In [ ]:
DAILY_ROWS = 50_000_000
BYTES_PER_ROW = 400
COMPRESSION = 4
OVERHEAD = 1.15
COPIES = 2
RETENTION_DAYS = 730

raw_gb_per_day = DAILY_ROWS * BYTES_PER_ROW / 1e9
stored_gb_per_day = raw_gb_per_day / COMPRESSION * OVERHEAD * COPIES
total_gb = stored_gb_per_day * RETENTION_DAYS

print(f"raw          = {raw_gb_per_day:.1f} GB/day")
print(f"stored       = {stored_gb_per_day:.1f} GB/day")
print(f"{RETENTION_DAYS} days     = {total_gb:.1f} GB = {total_gb / 1000:.3f} TB")

assert raw_gb_per_day == 20.0
assert stored_gb_per_day == 11.5
assert round(total_gb, 1) == 8395.0
assert round(total_gb / 1000, 3) == 8.395

### Try it 2. Compute transformation cost from data volume and engine rate

**Task.** A nightly transformation reads 2000 GB. Each of 8 worker nodes processes 5 GB per minute. Shuffle and write overhead adds 25 percent to the billed runtime, and a node-hour costs 6.00. Compute the full-refresh cost, then the cost when an incremental run reads only 5 percent of the data.

**Hint.** `runtime_minutes = total_gb / (rate_per_node * nodes)`. Billed node-hours are `nodes * runtime_minutes * overhead / 60`.

**Expected result.** Full refresh runs `50` minutes, bills `8.333` node-hours, and costs `50.0`. The incremental run costs `2.5`, a 20 times reduction.

In [ ]:
TOTAL_GB = 2000
NODES = 8
GB_PER_NODE_MINUTE = 5.0
OVERHEAD = 1.25
COST_PER_NODE_HOUR = 6.0


def transform_cost(gb, nodes=NODES):
    runtime_min = gb / (GB_PER_NODE_MINUTE * nodes)
    billed_min = runtime_min * OVERHEAD
    node_hours = nodes * billed_min / 60
    return runtime_min, node_hours, node_hours * COST_PER_NODE_HOUR


full_min, full_hours, full_cost = transform_cost(TOTAL_GB)
inc_min, inc_hours, inc_cost = transform_cost(TOTAL_GB * 0.05)

print(f"full refresh: {full_min:.0f} min, {full_hours:.3f} node-hours, {full_cost:.1f}")
print(f"incremental : {inc_min:.1f} min, {inc_hours:.3f} node-hours, {inc_cost:.1f}")

assert full_min == 50.0
assert round(full_hours, 3) == 8.333
assert full_cost == 50.0
assert inc_cost == 2.5
assert round(full_cost / inc_cost) == 20

### Try it 3. Compare batch and streaming freshness and cost

**Task.** A batch job runs every 60 minutes and costs 18.00 per run. A streaming job runs every 5 minutes and costs 5.00 per run. Assume stale data is, on average, half the interval. Compute the daily cost and average freshness of each, then the cost of each minute of freshness removed.

**Hint.** Runs per day are `1440 / interval_minutes`. Average latency is `interval_minutes / 2`. Divide the cost difference by the freshness difference.

**Expected result.** Batch costs `432.0` per day at `30.0` minutes average freshness. Streaming costs `1440.0` per day at `2.5` minutes, so each minute of freshness costs about `36.65` dollars per day.

In [ ]:
BATCH_INTERVAL = 60
BATCH_COST_PER_RUN = 18.0
STREAM_INTERVAL = 5
STREAM_COST_PER_RUN = 5.0
MINUTES_PER_DAY = 24 * 60


def profile(interval, cost_per_run):
    runs = MINUTES_PER_DAY / interval
    return runs, runs * cost_per_run, interval / 2


b_runs, b_cost, b_fresh = profile(BATCH_INTERVAL, BATCH_COST_PER_RUN)
s_runs, s_cost, s_fresh = profile(STREAM_INTERVAL, STREAM_COST_PER_RUN)
dollars_per_minute = (s_cost - b_cost) / (b_fresh - s_fresh)

print(f"batch    : {b_runs:.0f} runs, {b_cost:.1f}/day, {b_fresh:.1f} min fresh")
print(f"streaming: {s_runs:.0f} runs, {s_cost:.1f}/day, {s_fresh:.1f} min fresh")
print(f"freshness costs {dollars_per_minute:.2f} per minute per day")

assert b_cost == 432.0 and b_fresh == 30.0
assert s_cost == 1440.0 and s_fresh == 2.5
assert round(dollars_per_minute, 2) == 36.65

### Try it 4. Plan a backfill window from throughput

**Task.** You must rebuild 365 days for a table that grows by 50,000,000 rows per day. The end-to-end rate, including transform and load, is 12,000 rows per second per stream. Compute the serial wall-clock time, then the time when 6 streams run in parallel.

**Hint.** `seconds = total_rows / effective_rows_per_second`. Divide by the stream count or by the number of hours in the nightly window to get the practical schedule.

**Expected result.** About `1,520,833` seconds, or `422.45` hours serial, falling to about `70.41` hours or `2.93` days with 6 streams.

In [ ]:
HISTORY_DAYS = 365
ROWS_PER_DAY = 50_000_000
ROWS_PER_SECOND = 12_000
STREAMS = 6

total_rows = HISTORY_DAYS * ROWS_PER_DAY
serial_seconds = total_rows / ROWS_PER_SECOND
serial_hours = serial_seconds / 3600
parallel_hours = serial_hours / STREAMS

print(f"total rows     = {total_rows:,}")
print(f"serial         = {serial_seconds:,.0f} s = {serial_hours:.2f} h")
print(f"with {STREAMS} streams  = {parallel_hours:.2f} h = {parallel_hours / 24:.2f} days")

assert total_rows == 18_250_000_000
assert round(serial_seconds) == 1_520_833
assert round(serial_hours, 2) == 422.45
assert round(parallel_hours, 2) == 70.41
assert round(parallel_hours / 24, 2) == 2.93

### Try it 5. Compute SCD2 growth

**Task.** A customer dimension starts with 2,000,000 rows. Each year, 8 percent of the customers change an attribute that this dimension tracks, and each change adds a new version. Compute the row count after 5 years, the growth factor, and the storage needed at 600 bytes per row.

**Hint.** Extra versions after `n` years are `base * change_rate * n` when the change rate is applied to the base population each year. Storage is `rows * bytes_per_row`.

**Expected result.** `2,800,000` rows after 5 years, a growth factor of `1.4`, and `1.68` GB of dimension storage.

In [ ]:
BASE_ROWS = 2_000_000
ANNUAL_CHANGE_RATE = 0.08
YEARS = 5
BYTES_PER_ROW = 600

extra_versions = BASE_ROWS * ANNUAL_CHANGE_RATE * YEARS
rows_after = int(BASE_ROWS + extra_versions)
growth_factor = rows_after / BASE_ROWS
storage_gb = rows_after * BYTES_PER_ROW / 1e9

print(f"rows after {YEARS} years = {rows_after:,}")
print(f"growth factor          = {growth_factor:.2f}")
print(f"dimension storage      = {storage_gb:.2f} GB")

assert rows_after == 2_800_000
assert growth_factor == 1.4
assert storage_gb == 1.68

### Try it 6. Estimate concurrency needs from user counts and query time

**Task.** 400 analysts each run 6 queries per hour. At peak, activity triples. An average query occupies its slot for 20 seconds. Compute the query-seconds demanded per peak hour, the number of concurrently busy slots, and the number of 8-slot warehouses needed.

**Hint.** Demand in query-seconds is `users * queries_per_hour * peak_factor * seconds`. The slot count is that demand divided by 3600 seconds. Round warehouses up.

**Expected result.** `144,000` query-seconds per hour, `40.0` busy slots, and `5` warehouses of 8 slots.

In [ ]:
import math

USERS = 400
QUERIES_PER_USER_HOUR = 6
PEAK_FACTOR = 3
SECONDS_PER_QUERY = 20
SLOTS_PER_WAREHOUSE = 8

demand_seconds = USERS * QUERIES_PER_USER_HOUR * PEAK_FACTOR * SECONDS_PER_QUERY
busy_slots = demand_seconds / 3600
warehouses = math.ceil(busy_slots / SLOTS_PER_WAREHOUSE)

print(f"demand   = {demand_seconds:,} query-seconds per peak hour")
print(f"slots    = {busy_slots:.1f} concurrently busy")
print(f"warehouse= {warehouses} x {SLOTS_PER_WAREHOUSE} slots")

assert demand_seconds == 144_000
assert busy_slots == 40.0
assert warehouses == 5

### Try it 7. Compute a cost per report

**Task.** The warehouse costs 48,000 per month. Dashboard refresh consumes 20 percent of that. There are 1,500 reports, each refreshed once per day for 30 days. Compute the cost per refresh and the monthly cost per report.

**Hint.** Monthly refreshes are `reports * days`. Cost per refresh is the allocated spend divided by refreshes; cost per report is the allocated spend divided by reports.

**Expected result.** `45,000` refreshes, `0.213` per refresh, and `6.40` per report per month.

In [ ]:
MONTHLY_COST = 48_000.0
DASHBOARD_SHARE = 0.20
REPORTS = 1_500
DAYS = 30

dashboard_cost = MONTHLY_COST * DASHBOARD_SHARE
refreshes = REPORTS * DAYS
cost_per_refresh = dashboard_cost / refreshes
cost_per_report = dashboard_cost / REPORTS

print(f"dashboard spend  = {dashboard_cost:,.0f}/month")
print(f"refreshes        = {refreshes:,}")
print(f"cost per refresh = {cost_per_refresh:.3f}")
print(f"cost per report  = {cost_per_report:.2f}/month")

assert dashboard_cost == 9_600.0
assert refreshes == 45_000
assert round(cost_per_refresh, 3) == 0.213
assert round(cost_per_report, 2) == 6.40

### Try it 8. Detect an idempotency defect from a re-run scenario

**Task.** A pipeline appends one partition keyed by `order_id`. The loader produced 50,000 rows for the day. The task is retried after a transient failure and appends the same 50,000 rows again. Model the append, detect the duplicate keys, and show what the row count should be after a delete-then-insert re-run.

**Hint.** Build the partition as a list of keys. A second append doubles the list. The distinct count stays 50,000, so `len(partition) != len(set(partition))` is the defect signal.

**Expected result.** The partition holds `100,000` rows with `50,000` distinct keys, so the defect is `True`. The idempotent re-run leaves `50,000` rows and defect `False`.

In [ ]:
ROWS_IN_BATCH = 50_000

# Append-style load: the retry adds the same keys again.
partition = list(range(ROWS_IN_BATCH))
partition.extend(range(ROWS_IN_BATCH))

duplicates = len(partition) - len(set(partition))
defect = duplicates > 0
print(f"append rows    = {len(partition):,}")
print(f"distinct keys  = {len(set(partition)):,}")
print(f"duplicates     = {duplicates:,}, idempotency defect = {defect}")

assert len(partition) == 100_000
assert len(set(partition)) == 50_000
assert defect is True

# Idempotent load: delete the interval, then insert once.
partition = []
partition.extend(range(ROWS_IN_BATCH))
print(f"after idempotent re-run rows = {len(partition):,}")
assert len(partition) == 50_000
assert len(partition) == len(set(partition))
assert (len(partition) - len(set(partition))) == 0